# Order Book Management & Validation — Solberg Trading Desk

**Scenario:** Solberg Kraft AS runs a trading desk that aggregates bids across multiple assets
and submits them to Nord Pool for the Nordic day-ahead auction.

The desk needs to:
1. Collect bids from asset managers (wind, hydro, gas)
2. Organise them in an `OrderBook` portfolio container
3. Run EUPHEMIA validation before submission
4. Export to Nord Pool auction format
5. Generate summary reports for the desk

This notebook demonstrates:
- `OrderBook` creation and management (add, remove, filter, query)
- EUPHEMIA compliance validation via the `validation` module
- Exporting to a pandas DataFrame for reporting
- Nord Pool auction format conversion
- Portfolio summary visualisations

## Prerequisites

```bash
pip install nexa-bidkit matplotlib
# or, from source:
poetry install
```

In [ ]:
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import zoneinfo

import matplotlib.pyplot as plt
import pandas as pd

# Bid construction
from nexa_bidkit.bids import (
    simple_bid_from_curve,
    indivisible_block_bid,
    linked_block_bid,
    exclusive_group,
    SimpleBid,
    BlockBid,
    LinkedBlockBid,
)
# Curves
from nexa_bidkit.curves import from_dict_list, to_dataframe as curves_to_df
# Order book management
from nexa_bidkit.orders import (
    create_order_book,
    add_bids,
    filter_bids,
    get_bids_by_zone,
    get_bids_by_type,
    count_bids,
    total_volume_by_zone,
    get_order_book_summary,
    update_all_statuses,
    to_dataframe as orders_to_df,
)
# Validation
from nexa_bidkit.validation import (
    validate_order_book_for_submission,
    validate_order_book_bids,
    validate_bids,
    get_validation_summary,
    validate_simple_bid,
    EuphemiaValidationError,
    DataQualityError,
    ValidationError,
)
# Types
from nexa_bidkit.types import (
    BiddingZone,
    BidStatus,
    BidType,
    CurveType,
    DeliveryPeriod,
    Direction,
    MTUDuration,
    MTUInterval,
    PriceQuantityCurve,
    PriceQuantityStep,
)
# Nord Pool export
from nexa_bidkit.nordpool import order_book_to_nord_pool

CET = zoneinfo.ZoneInfo("Europe/Oslo")
print("nexa-bidkit imports OK")

## 1. Delivery Day Setup

In [ ]:
delivery_day = datetime(2026, 3, 15, tzinfo=CET)
MTU = MTUDuration.QUARTER_HOURLY

def mtu_at(hour: int, minute: int = 0) -> MTUInterval:
    """Create a single 15-min MTU at the given hour:minute on the delivery day."""
    return MTUInterval.from_start(
        start=delivery_day + timedelta(hours=hour, minutes=minute),
        duration=MTU,
    )

def period(h_start: int, h_end: int) -> DeliveryPeriod:
    """Create a DeliveryPeriod spanning whole hours on the delivery day."""
    return DeliveryPeriod(
        start=delivery_day + timedelta(hours=h_start),
        end=delivery_day + timedelta(hours=h_end),
        duration=MTU,
    )

def supply_curve(steps_data: list[tuple[float, float]], mtu: MTUInterval) -> PriceQuantityCurve:
    """Build a supply curve from (price_eur, volume_mw) tuples."""
    return from_dict_list(
        steps=[{"price": p, "volume": v} for p, v in steps_data],
        curve_type=CurveType.SUPPLY,
        mtu=mtu,
    )

print(f"Delivery day: {delivery_day.strftime('%Y-%m-%d')} (CET)")
print(f"MTU duration: {MTU.value} ({MTU.per_day} MTUs/day)")

## 2. Asset Manager Bids

Each asset manager submits their bids to the trading desk. We create representative
bids across the three main bid types.

In [ ]:
# --- Wind farm: 6 hourly simple bids (hours 6–12) ---
wind_forecast = [75.0, 80.0, 85.0, 88.0, 90.0, 87.0]  # MW per hour
wind_bids = []
for i, mw in enumerate(wind_forecast):
    hour = 6 + i
    # Use 4 MTUs per hour; just submit the first MTU as a representative example
    for m in range(4):
        mtu = mtu_at(hour, m * 15)
        curve = supply_curve([
            (5.0, mw * 0.4),
            (22.0, mw * 0.4),
            (48.0, mw * 0.2),
        ], mtu)
        wind_bids.append(simple_bid_from_curve(
            curve=curve,
            bidding_zone=BiddingZone.NO2,
            bid_id=f"wind-h{hour:02d}m{m:02d}",
            metadata={"asset": "solberg-wind"},
        ))

print(f"Wind bids created: {len(wind_bids)}")

# --- Run-of-river hydro: 4 hourly simple bids (hours 8–12) ---
ror_bids = []
for i in range(4):
    hour = 8 + i
    for m in range(4):
        mtu = mtu_at(hour, m * 15)
        curve = supply_curve([(3.0, 80.0), (10.0, 60.0), (18.0, 40.0)], mtu)
        ror_bids.append(simple_bid_from_curve(
            curve=curve,
            bidding_zone=BiddingZone.NO5,
            bid_id=f"ror-h{hour:02d}m{m:02d}",
            metadata={"asset": "voss-ror"},
        ))

print(f"RoR hydro bids created: {len(ror_bids)}")

# --- CCGT: main generation block + startup linked block ---
main_block = indivisible_block_bid(
    bidding_zone=BiddingZone.NO1,
    direction=Direction.SELL,
    delivery_period=period(8, 14),
    price=Decimal("65.00"),
    volume=Decimal("300"),
    bid_id="ccgt-main",
    metadata={"asset": "solberg-ccgt"},
)

startup_block = linked_block_bid(
    parent_bid_id="ccgt-main",
    bidding_zone=BiddingZone.NO1,
    direction=Direction.SELL,
    delivery_period=period(6, 8),
    price=Decimal("76.50"),  # marginal + startup recovery
    volume=Decimal("200"),
    bid_id="ccgt-startup",
    metadata={"asset": "solberg-ccgt", "cost_type": "startup"},
)

# --- Hydro reservoir: exclusive group (base-load vs peak-only) ---
hydro_base = indivisible_block_bid(
    bidding_zone=BiddingZone.NO1,
    direction=Direction.SELL,
    delivery_period=period(6, 14),
    price=Decimal("38.00"),
    volume=Decimal("250"),
    bid_id="hydro-base",
)

hydro_peak = indivisible_block_bid(
    bidding_zone=BiddingZone.NO1,
    direction=Direction.SELL,
    delivery_period=period(8, 12),
    price=Decimal("60.00"),
    volume=Decimal("300"),
    bid_id="hydro-peak",
)

hydro_group = exclusive_group(
    block_bids=[hydro_base, hydro_peak],
    group_id="hydro-modes",
    metadata={"asset": "rjukan-reservoir"},
)

print(f"\nBlock bids: main={main_block.bid_id}, startup={startup_block.bid_id}")
print(f"Exclusive group: {hydro_group.group_id} ({hydro_group.member_count} members)")

## 3. Build the Order Book

The `OrderBook` is an immutable container. All mutations return new instances.

In [ ]:
# Start with an empty book and add all bids
book = create_order_book(
    metadata={"desk": "solberg-trading", "delivery_date": "2026-03-15"}
)

# Add simple bids
book = add_bids(book, wind_bids)
book = add_bids(book, ror_bids)

# Add block / linked / exclusive bids
book = add_bids(book, [main_block, startup_block, hydro_group])

# Summary
summary = get_order_book_summary(book)
print("Order book summary:")
print(f"  ID:         {summary['order_book_id']}")
print(f"  Total bids: {summary['total_bids']}")
print(f"  Bid types:  {summary['bid_counts']}")
print(f"  Zones:      {summary['zones']}")
print(f"  Directions: {summary['directions']}")
print(f"  Statuses:   {summary['statuses']}")

## 4. Querying the Order Book

Filter and query bids by zone, type, or custom predicates.

In [ ]:
# Get bids by zone
no1_bids = get_bids_by_zone(book, BiddingZone.NO1)
no2_bids = get_bids_by_zone(book, BiddingZone.NO2)
no5_bids = get_bids_by_zone(book, BiddingZone.NO5)

print(f"Bids by zone: NO1={len(no1_bids)}, NO2={len(no2_bids)}, NO5={len(no5_bids)}")

# Get all block-type bids (BLOCK and LINKED_BLOCK)
block_bids = get_bids_by_type(book, BidType.BLOCK)
linked_bids = get_bids_by_type(book, BidType.LINKED_BLOCK)
exclusive_bids = get_bids_by_type(book, BidType.EXCLUSIVE_GROUP)

print(f"Block types: BLOCK={len(block_bids)}, LINKED={len(linked_bids)}, EXCLUSIVE={len(exclusive_bids)}")

# Filter using a custom predicate — e.g. only high-price bids (≥ EUR 60)
from nexa_bidkit.bids import BlockBid, LinkedBlockBid

high_price_book = filter_bids(
    book,
    lambda bid: (
        isinstance(bid, (BlockBid, LinkedBlockBid)) and float(bid.price) >= 60.0
    ) or bid.bid_type == BidType.SIMPLE_HOURLY
)

print(f"After high-price filter: {len(high_price_book.bids)} bids remain")

## 5. EUPHEMIA Validation

Before submitting, run validation to catch any EUPHEMIA compliance issues.

In [ ]:
# Run batch validation: get results for every bid
validation_results = validate_bids(list(book.bids))
summary = get_validation_summary(validation_results)

print("Validation summary:")
print(f"  Total bids: {summary['total_bids']}")
print(f"  Passed:     {summary['passed']}")
print(f"  Failed:     {summary['failed']}")
print(f"  Pass rate:  {summary['pass_rate']:.1f}%")
if summary['error_types']:
    print(f"  Error types: {summary['error_types']}")
else:
    print("  All bids passed validation!")

# Also validate gate closure (raises if submission is after gate closure)
gate_closure = datetime(2026, 3, 14, 12, 0, 0, tzinfo=CET)  # 12:00 CET on D-1
now = datetime(2026, 3, 14, 10, 30, 0, tzinfo=CET)           # current time (before gate)

try:
    validate_order_book_for_submission(book, gate_closure_time=gate_closure, submission_time=now)
    print(f"\nGate closure check PASSED (submission at {now.strftime('%H:%M')} < gate {gate_closure.strftime('%H:%M')} CET)")
except ValidationError as e:
    print(f"\nGate closure check FAILED: {e}")

In [ ]:
# Demonstrate catching a validation error on a dust bid (volume < 0.1 MW)
print("Testing validation catch on a dust bid (volume < 0.1 MW):")
try:
    dust_curve = PriceQuantityCurve(
        curve_type=CurveType.SUPPLY,
        steps=[PriceQuantityStep(price=Decimal("50"), volume=Decimal("0.05"))],
        mtu=mtu_at(9),
    )
    dust_bid = simple_bid_from_curve(
        curve=dust_curve,
        bidding_zone=BiddingZone.NO2,
        bid_id="dust-bid",
    )
    validate_simple_bid(dust_bid)
    print("  UNEXPECTED: bid passed validation")
except DataQualityError as e:
    print(f"  Caught DataQualityError: {e}")
except ValidationError as e:
    print(f"  Caught ValidationError: {e}")

## 6. Export to DataFrame

Export the order book to a pandas DataFrame for reporting and downstream processing.

In [ ]:
df = orders_to_df(book)
print(f"Order book DataFrame: {len(df)} rows × {len(df.columns)} columns")
print(f"\nBid type distribution:")
print(df["bid_type"].value_counts().to_string())
print(f"\nBidding zones:")
print(df["bidding_zone"].value_counts().to_string())

# Show block bid rows
block_df = df[df["bid_type"].isin(["BLOCK", "LINKED_BLOCK"])]
print(f"\nBlock bids:")
print(block_df[["bid_id", "bid_type", "bidding_zone", "price",
                "volume_per_mtu", "mtu_count", "parent_bid_id"]].to_string(index=False))

## 7. Update Status Before Submission

After validation passes, mark all bids as VALIDATED before submitting.

In [ ]:
# Transition from DRAFT → VALIDATED
validated_book = update_all_statuses(
    book,
    new_status=BidStatus.VALIDATED,
    filter_current_status=BidStatus.DRAFT,
)

status_counts_before = get_order_book_summary(book)["statuses"]
status_counts_after = get_order_book_summary(validated_book)["statuses"]

print(f"Status before update: {status_counts_before}")
print(f"Status after update:  {status_counts_after}")
print(f"\nOriginal book unchanged (immutable): {get_order_book_summary(book)['statuses']}")

## 8. Nord Pool Export

Convert simple bids to Nord Pool auction request format. Nord Pool requires a
`ContractIdResolver` callable to map MTU intervals to their contract ID strings
(these come from Nord Pool's products API — here we simulate one).

In [ ]:
def mock_contract_id_resolver(mtu: MTUInterval, zone: BiddingZone) -> str:
    """Simulate a Nord Pool contract ID resolver.

    In production this would call Nord Pool's products API to look up the
    contract ID for each MTU + bidding zone combination.
    """
    hour = mtu.start.astimezone(CET).hour
    minute = mtu.start.astimezone(CET).minute
    return f"{zone.value}-{hour:02d}{minute:02d}"

# Export a sample of NO2 simple bids to a small order book
no2_simple_bids = [
    b for b in validated_book.bids
    if isinstance(b, SimpleBid) and b.bidding_zone == BiddingZone.NO2
]
print(f"Total NO2 simple bids: {len(no2_simple_bids)}")

# Create a sample book with just 3 NO2 bids
sample_book = add_bids(create_order_book(), no2_simple_bids[:3])

np_submission = order_book_to_nord_pool(
    order_book=sample_book,
    auction_id="DA-2026-03-15",
    portfolio="solberg-trading",
    contract_id_resolver=mock_contract_id_resolver,
    comment="Day-ahead bids 2026-03-15",
)

print(f"\nNord Pool submission summary:")
print(f"  Curve orders: {len(np_submission.curve_orders)}")
print(f"  Block orders: {len(np_submission.block_orders)}")
print(f"\nFirst curve order:")
first = np_submission.curve_orders[0]
print(f"  Auction: {first.auction_id}, Portfolio: {first.portfolio}, Area: {first.area_code}")
print(f"  Curves (contracts): {len(first.curves)}")
for curve in first.curves[:2]:
    print(f"    Contract: {curve.contract_id}")
    for pt in curve.curve_points[:2]:
        print(f"      price={pt.price:.2f} EUR/MWh, volume={pt.volume:.1f} MW")

## 9. Portfolio Summary Visualisations

Visual overview of the order book: bid type distribution, volume by zone,
and bid status breakdown.

In [ ]:
summary = get_order_book_summary(validated_book)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f"Solberg Trading Desk — Order Book Summary ({delivery_day.strftime('%Y-%m-%d')})",
    fontsize=13, fontweight="bold"
)

# ---- Plot 1: Bid type distribution ----
bid_counts = {k: v for k, v in summary["bid_counts"].items() if v > 0}
type_colors = {"SIMPLE_HOURLY": "#3498db", "BLOCK": "#e67e22",
               "LINKED_BLOCK": "#e74c3c", "EXCLUSIVE_GROUP": "#9b59b6"}
axes[0].bar(
    [t.replace("_", "\n") for t in bid_counts.keys()],
    bid_counts.values(),
    color=[type_colors.get(t, "#95a5a6") for t in bid_counts.keys()],
    edgecolor="white"
)
axes[0].set_title("Bids by Type")
axes[0].set_ylabel("Count")
axes[0].grid(True, axis="y", alpha=0.3)
for bar, count in zip(axes[0].patches, bid_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(count), ha="center", va="bottom", fontsize=10, fontweight="bold")

# ---- Plot 2: Volume by bidding zone ----
vol_by_zone = total_volume_by_zone(validated_book)
zone_labels = [z.value for z in vol_by_zone.keys()]
zone_volumes = [float(v) for v in vol_by_zone.values()]
zone_colors = ["#27ae60", "#2980b9", "#f39c12"]

bars = axes[1].bar(zone_labels, zone_volumes, color=zone_colors[:len(zone_labels)], edgecolor="white")
axes[1].set_title("Total Bid Volume by Zone")
axes[1].set_ylabel("Volume (MW·MTUs)")
axes[1].grid(True, axis="y", alpha=0.3)
for bar, vol in zip(bars, zone_volumes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{vol:,.0f}", ha="center", va="bottom", fontsize=9)

# ---- Plot 3: Status donut ----
status_counts = {k: v for k, v in summary["statuses"].items() if v > 0}
status_colors = {"DRAFT": "#bdc3c7", "VALIDATED": "#2ecc71",
                 "SUBMITTED": "#3498db", "ACCEPTED": "#27ae60",
                 "REJECTED": "#e74c3c"}
wedge_colors = [status_colors.get(s, "#95a5a6") for s in status_counts.keys()]

wedges, texts, autotexts = axes[2].pie(
    status_counts.values(),
    labels=status_counts.keys(),
    colors=wedge_colors,
    autopct="%1.0f%%",
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor="white"),
)
axes[2].set_title("Bids by Status")

plt.tight_layout()
plt.savefig("order_book_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: order_book_summary.png")

## Summary

This notebook demonstrated end-to-end order book management for a trading desk:

| Step | API |
|------|-----|
| Create empty book | `create_order_book()` |
| Add bids | `add_bids(book, bids)` — immutable, returns new book |
| Query by zone/type | `get_bids_by_zone`, `get_bids_by_type` |
| Custom filter | `filter_bids(book, predicate)` |
| Validate | `get_validation_summary(book, gate_closure, now)` |
| Export to DataFrame | `to_dataframe(book)` (orders module) |
| Update status | `update_all_statuses(book, BidStatus.VALIDATED)` |
| Nord Pool export | `to_nordpool_request(book, contract_id_resolver)` |

Key design principles demonstrated:
- **Immutability**: Every mutation returns a new `OrderBook` — the original is unchanged
- **Type safety**: `Decimal` throughout prevents floating-point rounding in monetary values
- **Timezone awareness**: All datetimes are timezone-aware (no naive datetimes)
- **Separation of concerns**: Bid construction, validation, and exchange formatting are
  independent modules that compose cleanly